# Unify rasters to a base raster
including
- Reprojection
- Resampling
- Clipping

in one step.

In [1]:
# Standard libraries
import rasterio
import os
from importlib.resources import files
from rasterio.enums import Resampling
from pathlib import Path
from tqdm import tqdm

# Custom modules
from beak.experimental.utilities.io import save_raster, create_file_folder_list, create_file_list, check_path
from beak.experimental.utilities.raster_processing import unify_raster_grids

In [9]:
BASE_PATH = files("beak.data")

BASE_NAME = "GEOPHYSICS_ISOGRAVITY"
BASE_SPATIAL = "EPSG_4326_RES_0_025"
BASE_EXTENT = "US_CANADA"

# Define input folder
input_folder = BASE_PATH / BASE_NAME / "RAW" / "USC"

# Export to current location for testing purposes
CURRENT_DIR = Path(os.getcwd())
PATH_EXPORT = CURRENT_DIR / "PROCESSED" 

# Set output folder
output_folder = PATH_EXPORT / BASE_SPATIAL / BASE_EXTENT / "UNIFIED"

# Set base raster: In this case, we use the base raster created with the Lawley (2022) datacube
base_raster = BASE_PATH / "BASE_RASTERS" / str(BASE_SPATIAL + "_" + BASE_EXTENT + ".tif")

print("Input folder: ", input_folder)
print("Base raster: ", base_raster)
print("Output folder: ", output_folder)

Input folder:  s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\GEOPHYSICS_ISOGRAVITY\RAW\USC
Base raster:  s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\BASE_RASTERS\EPSG_4326_RES_0_025_US_CANADA.tif
Output folder:  s:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\examples\notebooks\dataset_isostatic_gravity\PROCESSED\EPSG_4326_RES_0_025\US_CANADA\UNIFIED


In [4]:
# Create file and folder list
folders, _ = create_file_folder_list(input_folder)
folders.insert(0, input_folder)

file_list = []
for folder in folders:
  folder_files = create_file_list(folder, recursive=True)
  file_list.extend(folder_files)
  
print(f"Found {len(file_list)} files in {len(folders)} folders:")

for file in file_list:
  print(file)

Found 3 files in 1 folders:
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\GEOPHYSICS_ISOGRAVITY\RAW\USC\US_IsostaticGravity_HGM_WGS84.tif
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\GEOPHYSICS_ISOGRAVITY\RAW\USC\US_IsostaticGravity_HGM_WGS84_Worms.tif
s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\GEOPHYSICS_ISOGRAVITY\RAW\USC\US_IsostaticGravity_WGS84.tif


In [5]:
# Select to write output file
dry_run = False
if dry_run:
  print("Dry run, no files will be written.\n")
  
# Unify  
for file in tqdm(file_list, total=len(file_list)):
  out_path = output_folder / file.relative_to(input_folder)
  check_path(Path(os.path.dirname(out_path)))
  
  raster = rasterio.open(file)
  unified_raster, meta = unify_raster_grids(base_raster, [file], resampling_method=Resampling.bilinear, same_extent=True, same_shape=True)[0]

  if not dry_run:
    save_raster(out_path,
          array=unified_raster,
          crs=meta['crs'],
          height=meta['height'],
          width=meta['width'],
          nodata_value=meta['nodata'],
          transform=meta['transform'],
          dtype="float32")


100%|██████████| 3/3 [00:01<00:00,  1.70it/s]
